# Enterprise RAG OS
**Built by:** Ravi Gohel | B.Tech AI/ML, Rajkot, Gujarat

**GitHub:** github.com/ravigohel142996/enterprise-rag-os

**Stack:** ChromaDB + LLaMA 3 (Groq) + Python

---
## What this does
- Upload any document → Ask any question → Get a **grounded, cited answer**
- **Zero hallucination** — LLM answers ONLY from your documents
- 100% free tier — no paid APIs, no cloud cost

---
## Setup
Get your FREE Groq API key at: https://console.groq.com

In [ ]:
# Cell 1 — Install libraries
# Run this once, then Runtime > Restart session
!pip install chromadb groq -q
print('Done! Now go to Runtime > Restart session, then run Cell 2 onwards.')

In [ ]:
# Cell 2 — Setup ChromaDB + load documents
# API key is entered securely at runtime — never saved in notebook
import chromadb
from chromadb.utils import embedding_functions
from getpass import getpass

# Securely enter your Groq API key (not saved in notebook)
# Get free key: https://console.groq.com -> API Keys -> Create Key
GROQ_API_KEY = getpass('Enter your Groq API key: ')

# ChromaDB built-in embedding — lightweight, no RAM crash
emb_fn = embedding_functions.DefaultEmbeddingFunction()

client = chromadb.Client()
try:
    client.delete_collection('enterprise_rag')
except Exception:
    pass
collection = client.create_collection(
    name='enterprise_rag',
    embedding_function=emb_fn
)

# Your documents — replace with your own!
documents = [
    'Ravi Gohel is a B.Tech AI/ML student from Rajkot, Gujarat.',
    'The company refund policy allows returns within 30 days of purchase.',
    'RAG stands for Retrieval-Augmented Generation — it combines search with LLMs.',
    'Python is used for AI/ML development with TensorFlow and PyTorch.',
    'ChromaDB is an open-source vector database for AI applications.',
    'The leave policy grants 12 sick days and 15 casual leaves per year.',
    'FastAPI is a modern Python web framework for building REST APIs.',
    'Enterprise RAG OS is a production-grade RAG system built by Ravi Gohel.',
    'The project supports PDF, CSV, and TXT document formats.',
    'Vector embeddings convert text into numbers that capture semantic meaning.',
]

collection.add(
    documents=documents,
    ids=[f'doc_{i}' for i in range(len(documents))]
)
print(f'Stored {len(documents)} documents in ChromaDB!')

In [ ]:
# Cell 3 — Core RAG pipeline
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

def rag_query(user_question, top_k=3):
    print(f'\n{"="*55}')
    print(f'Question: {user_question}')
    print('='*55)

    results = collection.query(
        query_texts=[user_question],
        n_results=top_k
    )
    retrieved_chunks = results['documents'][0]

    print('\nRetrieved context:')
    for i, chunk in enumerate(retrieved_chunks):
        print(f'  [{i+1}] {chunk}')

    context = '\n'.join(retrieved_chunks)
    prompt = f"""You are a helpful assistant. Answer ONLY using the context below.
If the answer is not in the context, say exactly: I don't have that information.
Do not make up any information.

Context:
{context}

Question: {user_question}
Answer:"""

    response = groq_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=300,
        temperature=0.1
    )

    answer = response.choices[0].message.content.strip()
    print(f'\nAnswer: {answer}')
    return answer

print('RAG pipeline READY!')

In [ ]:
# Cell 4 — Live Demo
print('Enterprise RAG OS — Live Demo')
print('='*55)

rag_query('What is the refund policy?')
rag_query('How many sick days do employees get?')
rag_query('What is RAG?')
rag_query('Who is Ravi Gohel?')

# Hallucination test — proves zero hallucination
print('\n--- HALLUCINATION TEST ---')
rag_query('What is the capital of France?')
rag_query('What is the stock price of Apple?')

In [ ]:
# Cell 5 (BONUS) — Interactive chat loop
# Type your own questions live!
print('Interactive RAG Chat')
print('Type quit to exit\n')
while True:
    user_input = input('You: ').strip()
    if user_input.lower() in ['quit', 'exit', 'q']:
        print('Session ended.')
        break
    if user_input:
        rag_query(user_input)